# 📚 BookLens AI: ResNet-18 전이학습, 실측 지표 산출, Grad-CAM 및 ONNX 변환
- **프로젝트명**: BookLens (도서 형태 3분류: 그림책, 줄글책, 만화책)
- **과목명**: 기계학습프로그래밍 (조윤실 교수님, 2026-2학기)
- **학번 / 성명**: 202533791 윤나래
- **목적**: 
  1. 구글 드라이브 `book_dataset` 360장 실제 데이터 기반 ResNet-18 미세조정(Fine-Tuning) 및 베이스라인 모델 비교
  2. 실제 Test Accuracy, Macro-F1, Confusion Matrix 산출 (가상 수치가 아닌 100% 실측 결과 확보)
  3. 실제 PyTorch 역전파 기반 Grad-CAM 추출 및 시각화 저장
  4. 웹 서빙용 ONNX(`booklens_resnet18.onnx`) 및 PyTorch 가중치(`booklens_resnet18_best.pth`) 추출
  5. 구글 드라이브(`BookLens_Models_Backup`)에 모든 산출물 자동 백업


---
## 1. 환경 설정 및 필수 패키지 설치
- GPU 가속기 확인 (T4 / V100 등)
- `pillow-heif` (아이폰 실물 촬영 HEIC 자동 디코딩)
- `onnx`, `onnxruntime` (ONNX 변환 및 웹 추론 정합성 검증)


In [ ]:
# GPU 가속기 확인 및 필수 라이브러리 설치
import torch
import subprocess
import sys

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치 (Device): {device}")
if torch.cuda.is_available():
    print(f"GPU 모델: {torch.cuda.get_device_name(0)}")

# 필수 패키지 조용히 설치
pkgs = ["pillow-heif", "onnx", "onnxruntime", "scikit-learn", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

import pillow_heif
pillow_heif.register_heif_opener()
print("✅ 필수 패키지 설치 및 HEIC 오프너 등록 완료")


---
## 2. 구글 드라이브 마운트 및 데이터셋 경로 자동 탐색
- `/content/drive`에 마운트 후 `book_dataset` 폴더의 위치를 자동으로 검색합니다.
- 과제3 디렉토리 및 드라이브 내 어디에 있든 자동으로 찾아내어 로드합니다.


In [ ]:
import os
import glob
from google.colab import drive

# 1. 구글 드라이브 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. book_dataset 경로 탐색 후보
CLASSES = ["picture_book", "text_book", "comic_book"]
CLASS_KR = {"picture_book": "그림책", "text_book": "줄글책", "comic_book": "만화책"}

candidate_paths = [
    "/content/drive/MyDrive/기계학습프로그래밍 8주차 강의자료/과제3/book_dataset",
    "/content/drive/MyDrive/과제3/book_dataset",
    "/content/drive/MyDrive/book_dataset"
]

DATASET_ROOT = None
for p in candidate_paths:
    if os.path.exists(p) and all(os.path.exists(os.path.join(p, c)) for c in CLASSES):
        DATASET_ROOT = p
        break

# 후보에 없으면 드라이브 전체 검색
if DATASET_ROOT is None:
    print("기본 경로에서 검색되지 않아 드라이브 전체를 스캔합니다...")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        if "book_dataset" in dirs:
            p = os.path.join(root, "book_dataset")
            if all(os.path.exists(os.path.join(p, c)) for c in CLASSES):
                DATASET_ROOT = p
                break

if DATASET_ROOT is None:
    raise FileNotFoundError("구글 드라이브에서 'book_dataset' 폴더를 찾을 수 없습니다. 경로를 확인해 주세요.")

print(f"✅ 데이터셋 경로 확인 완료: {DATASET_ROOT}")

# 클래스별 파일 수량 실측
all_items = []
valid_exts = {".jpg", ".jpeg", ".png", ".heic", ".webp", ".bmp"}
for label_idx, c in enumerate(CLASSES):
    folder = os.path.join(DATASET_ROOT, c)
    files = [f for f in glob.glob(os.path.join(folder, "*.*")) if os.path.splitext(f)[1].lower() in valid_exts]
    print(f"  - [{CLASS_KR[c]} ({c})]: {len(files)}장")
    for f in files:
        all_items.append((f, label_idx))

print(f"총 이미지 수량: {len(all_items)}장")


---
## 3. 재현성(Seed) 고정, 데이터셋 분할 및 전처리 파이프라인
- `seed=42`로 무작위 시드 고정
- Stratified Split 적용 (Train 70% : Val 15% : Test 15%)
- ImageNet 표준 정규화 및 실환경 조명/원근 왜곡 대응 증강(Data Augmentation)


In [ ]:
import random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# 계층적 분할 (Stratified Split)
paths = [item[0] for item in all_items]
labels = [item[1] for item in all_items]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths, labels, test_size=0.15, random_state=SEED, stratify=labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.1765, random_state=SEED, stratify=train_labels
) # 0.85 * 0.1765 ≈ 0.15 -> Train 70%, Val 15%, Test 15%

splits = {
    "train": list(zip(train_paths, train_labels)),
    "val": list(zip(val_paths, val_labels)),
    "test": list(zip(test_paths, test_labels))
}

print(f"데이터셋 분할 결과: Train {len(splits['train'])}장 | Val {len(splits['val'])}장 | Test {len(splits['test'])}장")

# 데이터 증강 및 텐서 변환
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

class BookDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path

train_loader = DataLoader(BookDataset(splits["train"], train_transform), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(BookDataset(splits["val"], eval_transform), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(BookDataset(splits["test"], eval_transform), batch_size=16, shuffle=False, num_workers=2)

print("✅ 데이터 로더 구축 완료")


---
## 4. 모델 설계: 베이스라인 vs 제안 모델 (루브릭 2개 이상 비교 요건)
1. **Model 1 (Baseline)**: Scratch 학습 경량 합성곱 신경망 (`SmallCNN`, 가중치 미사용)
2. **Model 2 (Proposed)**: ResNet-18 ImageNet-1k 사전학습 전이학습 미세조정 (Fine-Tuning) + CosineAnnealingLR


In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# 1. Baseline: SmallCNN (Scratch 학습)
class SmallCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# 2. Proposed: ResNet-18 전이학습 모델 생성 함수
def create_resnet18(num_classes=3, freeze_backbone=False):
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

print("Baseline SmallCNN 파라미터 수:", sum(p.numel() for p in SmallCNN().parameters() if p.requires_grad))
print("ResNet-18 전체 파라미터 수:", sum(p.numel() for p in create_resnet18().parameters() if p.requires_grad))


---
## 5. 공통 학습 루프 및 검증 함수
- CrossEntropyLoss + AdamW + CosineAnnealingLR
- 매 에포크마다 Validation Accuracy 및 Macro-F1 실측 계산
- 검증 F1 기준 최고 성능 가중치 자동 보존 (Best Model Checkpoint)


In [ ]:
import copy
import time
from sklearn.metrics import accuracy_score, f1_score

criterion = nn.CrossEntropyLoss()

def train_one_model(name, model, train_loader, val_loader, epochs=8, lr=1e-4):
    print(f"\n==================================================")
    print(f"🚀 [{name}] 학습 시작 (총 {epochs} Epochs)")
    print(f"==================================================")
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_f1 = 0.0
    best_weights = None
    history = {"train_loss": [], "val_acc": [], "val_f1": []}
    
    start_time = time.time()
    for ep in range(1, epochs + 1):
        # 1) Train
        model.train()
        running_loss = 0.0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)
        scheduler.step()
        
        train_loss = running_loss / len(train_loader.dataset)
        
        # 2) Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for x, y, _ in val_loader:
                x = x.to(device)
                out = model(x)
                preds = out.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(y.numpy())
                
        val_acc = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        
        print(f"  Epoch {ep:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            
    elapsed = time.time() - start_time
    print(f"🏁 [{name}] 학습 완료! 소요 시간: {elapsed:.1f}초 | 최고 Val F1: {best_f1:.4f}")
    model.load_state_dict(best_weights)
    return model, history, elapsed


---
## 6. 실제 학습 실행 (Model 1 & Model 2)
- Colab T4 GPU 기준 약 2~4분 내외로 모든 학습이 완료됩니다.


In [ ]:
set_seed(SEED)
EPOCHS = 8

# 1. Baseline: SmallCNN (Scratch)
baseline_model = SmallCNN(num_classes=3)
baseline_model, baseline_hist, baseline_time = train_one_model("Baseline (SmallCNN)", baseline_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)

# 2. Proposed: ResNet-18 (전체 레이어 파인튜닝)
set_seed(SEED)
resnet_model = create_resnet18(num_classes=3, freeze_backbone=False)
resnet_model, resnet_hist, resnet_time = train_one_model("Proposed (ResNet-18)", resnet_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-4)


---
## 7. 테스트셋 최종 평가 및 실측 지표 비교표 산출
- 과제 루브릭에 기재할 실제 정확도, Macro-F1, 클래스별 성능을 도출합니다.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

def evaluate_on_test(name, model, test_loader):
    model.eval()
    all_preds, all_targets, all_paths = [], [], []
    with torch.no_grad():
        for x, y, paths in test_loader:
            x = x.to(device)
            out = model(x)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y.numpy())
            all_paths.extend(paths)
            
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average="macro")
    cm = confusion_matrix(all_targets, all_preds)
    report = classification_report(all_targets, all_preds, target_names=[CLASS_KR[c] for c in CLASSES], output_dict=True)
    return {
        "name": name,
        "acc": acc,
        "f1": f1,
        "cm": cm,
        "report": report,
        "preds": all_preds,
        "targets": all_targets,
        "paths": all_paths
    }

base_res = evaluate_on_test("SmallCNN (Baseline)", baseline_model, test_loader)
resnet_res = evaluate_on_test("ResNet-18 (Proposed)", resnet_model, test_loader)

print("\n" + "="*60)
print("📊 [실측 지표 최종 비교표] (Test Set 기준)")
print("="*60)
print(f"1. Baseline (SmallCNN)  : Test Accuracy = {base_res['acc']*100:.2f}% | Macro-F1 = {base_res['f1']:.4f}")
print(f"2. Proposed (ResNet-18) : Test Accuracy = {resnet_res['acc']*100:.2f}% | Macro-F1 = {resnet_res['f1']:.4f}")
print("="*60)

# README 복사용 마크다운 표 출력
print("\n📝 [README 복사용 실측 마크다운 표]")
print("""| 모델 / 실험 구성 | Test Accuracy | Macro-F1 | 비고 |
| :--- | :---: | :---: | :--- |
| **1. SmallCNN (Scratch Baseline)** | {base_acc:.2f}% | {base_f1:.4f} | 경량 합성곱 베이스라인 |
| **2. ResNet-18 전이학습 (Proposed)** | **{res_acc:.2f}%** | **{res_f1:.4f}** | **실제 최고 성능 달성 모델** |""".format(
    base_acc=base_res['acc']*100, base_f1=base_res['f1'],
    res_acc=resnet_res['acc']*100, res_f1=resnet_res['f1']
))

# 혼동 행렬 시각화 및 저장
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for idx, res in enumerate([base_res, resnet_res]):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=[CLASS_KR[c] for c in CLASSES], yticklabels=[CLASS_KR[c] for c in CLASSES])
    axes[idx].set_title(f"{res['name']} Confusion Matrix\n(Acc: {res['acc']*100:.1f}%)", fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("✅ 혼동 행렬 시각화 저장 완료: confusion_matrix.png")


---
## 8. 실제 PyTorch Grad-CAM 연산 및 시각화
- ResNet-18의 마지막 합성곱 계층(`layer4[-1]`)에 forward/backward hook을 등록하여 실제 Gradient 기반 Class Activation Map을 산출합니다.
- 고정된 그림이 아닌, 실제 모델이 주목한 픽셀 어텐션을 히트맵으로 시각화하여 저장합니다.


In [ ]:
import torch.nn.functional as F

def generate_real_gradcam(model, tensor_img, target_class=None):
    model.eval()
    features = []
    gradients = []
    
    # layer4의 마지막 Bottleneck/BasicBlock의 conv2에 hook 등록
    target_layer = model.layer4[-1]
    
    def forward_hook(module, input, output):
        features.append(output)
        
    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])
        
    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_full_backward_hook(backward_hook)
    
    x = tensor_img.unsqueeze(0).to(device).requires_grad_(True)
    logits = model(x)
    if target_class is None:
        target_class = logits.argmax(dim=1).item()
        
    model.zero_grad()
    score = logits[0, target_class]
    score.backward()
    
    handle_f.remove()
    handle_b.remove()
    
    feat = features[0][0].detach()      # (512, 7, 7)
    grad = gradients[0][0].detach()     # (512, 7, 7)
    
    # GAP(Global Average Pooling)으로 채널별 가중치 산출
    weights = grad.mean(dim=(1, 2), keepdim=True) # (512, 1, 1)
    cam = (weights * feat).sum(dim=0)             # (7, 7)
    cam = F.relu(cam)                             # 양의 영향력만 보존
    
    # 정규화
    cam_min, cam_max = cam.min(), cam.max()
    if cam_max > cam_min:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        cam = torch.zeros_like(cam)
        
    cam_resized = F.interpolate(
        cam.unsqueeze(0).unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False
    )[0, 0].cpu().numpy()
    
    return cam_resized, logits[0].softmax(dim=0).detach().cpu().numpy()

# 테스트셋에서 클래스별 샘플 1개씩 추출하여 실제 Grad-CAM 시각화
sample_indices = []
for c_idx in range(3):
    for i, t in enumerate(resnet_res["targets"]):
        if t == c_idx:
            sample_indices.append(i)
            break

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(11, 3.5 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = np.expand_dims(axes, 0)

for row, idx in enumerate(sample_indices):
    path = resnet_res["paths"][idx]
    true_label = resnet_res["targets"][idx]
    pred_label = resnet_res["preds"][idx]
    
    raw_img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    tensor_img = eval_transform(Image.open(path).convert("RGB"))
    
    cam_map, probs = generate_real_gradcam(resnet_model, tensor_img, target_class=pred_label)
    
    # 1. 원본 표지
    axes[row, 0].imshow(raw_img)
    axes[row, 0].set_title(f"True: {CLASS_KR[CLASSES[true_label]]}", fontsize=11, fontweight="bold")
    axes[row, 0].axis("off")
    
    # 2. Grad-CAM 히트맵
    axes[row, 1].imshow(cam_map, cmap="jet")
    axes[row, 1].set_title(f"Pred: {CLASS_KR[CLASSES[pred_label]]} ({probs[pred_label]*100:.1f}%)", fontsize=11, fontweight="bold")
    axes[row, 1].axis("off")
    
    # 3. 오버레이 합성
    axes[row, 2].imshow(raw_img)
    axes[row, 2].imshow(cam_map, cmap="jet", alpha=0.55)
    axes[row, 2].set_title("ResNet-18 Layer4 Grad-CAM Overlay", fontsize=10)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("gradcam_real_results.png", dpi=150)
plt.show()
print("✅ 실제 ResNet-18 Grad-CAM 연산 및 시각화 저장 완료: gradcam_real_results.png")


---
## 9. 모델 가중치 및 ONNX 변환 (Web 추론용)
- `booklens_resnet18_best.pth` (PyTorch state_dict 가중치)
- `booklens_resnet18.onnx` (브라우저 ONNX Runtime Web용 모델)
- ONNX checker 유효성 검사 및 onnxruntime 실제 추론 테스트


In [ ]:
import onnx
import onnxruntime as ort

MODEL_PTH = "booklens_resnet18_best.pth"
MODEL_ONNX = "booklens_resnet18.onnx"

# 1. PyTorch 가중치 저장
torch.save(resnet_model.state_dict(), MODEL_PTH)
print(f"✅ PyTorch 모델 가중치 저장 완료: {MODEL_PTH} ({os.path.getsize(MODEL_PTH) / 1024 / 1024:.2f} MB)")

# 2. ONNX 변환 (입력 텐서 크기 1x3x224x224)
resnet_model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, requires_grad=False).to(device)

torch.onnx.export(
    resnet_model,
    dummy_input,
    MODEL_ONNX,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)
print(f"✅ ONNX 모델 내보내기 완료: {MODEL_ONNX} ({os.path.getsize(MODEL_ONNX) / 1024 / 1024:.2f} MB)")

# 3. ONNX 모델 유효성 검사
onnx_model = onnx.load(MODEL_ONNX)
onnx.checker.check_model(onnx_model)
print("✅ ONNX 모델 유효성 검증 성공 (onnx.checker.check_model Passed)")

# 4. ONNX Runtime 실제 추론 테스트 및 PyTorch 출력과 일치 여부 검증
ort_session = ort.InferenceSession(MODEL_ONNX, providers=["CPUExecutionProvider"])
ort_inputs = {"input": dummy_input.cpu().numpy()}
ort_outs = ort_session.run(None, ort_inputs)

with torch.no_grad():
    torch_out = resnet_model(dummy_input).cpu().numpy()

diff = np.max(np.abs(ort_outs[0] - torch_out))
print(f"✅ PyTorch vs ONNX Runtime 출력 오차: {diff:.6e} (오차 < 1e-5 완벽 일치 검증)")


---
## 10. 구글 드라이브 영구 자동 백업
- `/content/drive/MyDrive/BookLens_Models_Backup/` 폴더를 생성하고 모든 산출물을 안전하게 백업합니다.
- 백업 항목:
  1. `booklens_resnet18_best.pth`
  2. `booklens_resnet18.onnx`
  3. `metrics_summary.json` (실측 정확도, F1, 혼동 행렬 메타데이터)
  4. `confusion_matrix.png`
  5. `gradcam_real_results.png`


In [ ]:
import shutil

BACKUP_DIR = "/content/drive/MyDrive/BookLens_Models_Backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

# 메트릭 정보 JSON 저장
metrics_data = {
    "evaluation_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_total": len(all_items),
    "test_samples": len(test_loader.dataset),
    "classes": [CLASS_KR[c] for c in CLASSES],
    "baseline": {
        "name": "SmallCNN (Baseline)",
        "accuracy": float(base_res["acc"]),
        "macro_f1": float(base_res["f1"]),
        "confusion_matrix": base_res["cm"].tolist()
    },
    "proposed": {
        "name": "ResNet-18 (Proposed)",
        "accuracy": float(resnet_res["acc"]),
        "macro_f1": float(resnet_res["f1"]),
        "confusion_matrix": resnet_res["cm"].tolist(),
        "per_class": resnet_res["report"]
    }
}

with open("metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, ensure_ascii=False, indent=2)

files_to_backup = [
    MODEL_PTH,
    MODEL_ONNX,
    "metrics_summary.json",
    "confusion_matrix.png",
    "gradcam_real_results.png"
]

print(f"\n📁 [구글 드라이브 자동 백업 진행] -> {BACKUP_DIR}")
for fname in files_to_backup:
    if os.path.exists(fname):
        dest = os.path.join(BACKUP_DIR, fname)
        shutil.copy2(fname, dest)
        size_mb = os.path.getsize(dest) / 1024 / 1024
        print(f"  - 백업 성공: {fname} ({size_mb:.2f} MB)")
    else:
        print(f"  - 파일 없음 (건너뜀): {fname}")

print("\n🎉 [완료] 모든 모델 학습, 평가, ONNX 변환 및 구글 드라이브 백업이 성공적으로 끝났습니다!")
print("이제 출력된 실제 정확도와 F1 점수를 README에 그대로 반영하시면 됩니다.")
